# Whisper Speech Recognition Example

This notebook demonstrates how to use the JAX NNX implementation of Whisper for speech recognition.

## Features
- Pure JAX implementation with no HuggingFace dependencies
- Loads pretrained Whisper weights
- Performs end-to-end speech transcription
- Uses Whisper's exact preprocessing pipeline


In [ ]:
import os
import time
import json
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np
import librosa
from flax import nnx
from huggingface_hub import snapshot_download

# Import our Whisper implementation
from bonsai.models.whisper import modeling, params


In [ ]:
# Load and generate transcription with our JAX NNX Whisper model

# Load audio file
audio_path = Path("audio_samples/bush_speech.wav")
print(f"Loading audio from: {audio_path}")

# Load and preprocess audio
audio, _ = librosa.load(str(audio_path), sr=16000)
print(f"Audio duration: {len(audio)/16000:.2f} seconds")

# Extract mel features using Whisper's exact preprocessing
mel_spec = librosa.feature.melspectrogram(
    y=audio, sr=16000, n_mels=80, hop_length=160, win_length=400,
    window='hann', fmin=0, fmax=8000, power=2.0
)

# Convert to log10 scale and normalize (Whisper's approach)
log_spec = np.log10(mel_spec + 1e-10)
log_spec = np.maximum(log_spec, log_spec.max() - 8.0)
log_spec = (log_spec + 4.0) / 4.0

mel_features = jnp.array(log_spec.T)  # Transpose to (time, n_mels)
print(f"Mel features shape: {mel_features.shape}")

# Pad to model's expected length
max_time_steps = 3000
if mel_features.shape[0] > max_time_steps:
    mel_features = mel_features[:max_time_steps]
else:
    padding = jnp.zeros((max_time_steps - mel_features.shape[0], mel_features.shape[1]))
    mel_features = jnp.concatenate([mel_features, padding], axis=0)

# Add batch dimension and transpose to (batch, n_mels, time)
mel_features = mel_features[None, ...].transpose(0, 2, 1)
print(f"Model input shape: {mel_features.shape}")


In [ ]:
# Load pretrained Whisper model
model_name = "openai/whisper-tiny"
MODEL_CP_PATH = "/tmp/models-bonsai/whisper-tiny"

# Download model if not present
if not os.path.isdir(MODEL_CP_PATH):
    print(f"Downloading {model_name} to {MODEL_CP_PATH}...")
    snapshot_download(model_name, local_dir=MODEL_CP_PATH)

# Create model from pretrained weights
print("Loading pretrained Whisper model...")
config = modeling.WhisperConfig.whisper_tiny()
model = params.create_model_from_safe_tensors(MODEL_CP_PATH, config)
print("✅ Model loaded successfully!")


In [ ]:
# Generate transcription
print("Generating transcription...")
start_time = time.time()

# Generate tokens
generated_tokens = modeling.generate(model, mel_features, max_length=100, temperature=0.0)

generation_time = time.time() - start_time
print(f"✅ Generated {len(generated_tokens[0])} tokens in {generation_time:.3f}s")

# Load vocabulary and decode tokens
vocab_path = os.path.join(MODEL_CP_PATH, "vocab.json")
with open(vocab_path, 'r') as f:
    vocab_data = json.load(f)

# Create reverse mapping: token_id -> text
vocab = {int(token_id): text for text, token_id in vocab_data.items()}

# Add special tokens
special_tokens = {
    50258: "<|startoftranscript|>", 50259: "<|en|>", 
    50359: "<|transcribe|>", 50363: "<|notimestamps|>", 50257: "<|endoftext|>"
}
vocab.update(special_tokens)

# Decode tokens to text
decoded_text = ""
for token in generated_tokens[0]:
    token_id = int(token)
    if token_id in vocab:
        text = vocab[token_id]
        # Skip special tokens for clean output
        if not (text.startswith("<|") and text.endswith("|>")):
            text = text.replace("Ġ", " ")  # Replace BPE space marker
            decoded_text += text

print(f"✅ Transcription: {decoded_text.strip()}")
